# Discussion 9: The Ray Ecosystem (Core, Data, and Train)
**DSC 232R - Big Data Analysis Using Spark**

Welcome to Week 9! While we've spent the quarter mastering Spark for ETL and data warehousing, today we are exploring **Ray**. Ray is a framework specifically purpose-built for scaling Python and compute-intensive machine learning workloads.

**Today's Agenda:**
1. **Ray Setup & Architecture:** Understanding the Ray cluster.
2. **Ray Core:** Distributed primitives (Tasks, Actors, and the Object Store).
3. **Ray Data:** Efficient, vectorized preprocessing for ML datasets.
4. **Ray Train:** Distributed model training with XGBoost.
5. **End-to-End Exercise:** Building a complete ML pipeline.

In [ ]:
# 1. Install required dependencies (Pinning pandas to <3 to prevent Ray Data errors)
# Note: This might take a minute or two.
!pip install -q -U "ray[default]" "ray[data]" "ray[train]" xgboost scikit-learn "pandas<3.0.0" numpy

# 2. (Optional) Access the Ray Dashboard in Colab to monitor cluster health
from google.colab import output
output.serve_kernel_port_as_window(8265)

Try `serve_kernel_port_as_iframe` instead. 


<IPython.core.display.Javascript object>



---
## Part 1: Ray Setup and Core Primitives

Ray Core provides three main pillars for distributed computing:
* **Tasks** (`@ray.remote` on functions): Stateless, parallel execution.
* **Actors** (`@ray.remote` on classes): Stateful, object-oriented execution.
* **Object Store** (`ray.put` / `ray.get`): Efficient, zero-copy memory sharing across tasks.

In [ ]:
import ray
import time
import numpy as np
import pandas as pd

# Initialize a local Ray cluster
if ray.is_initialized():
    ray.shutdown()

# We limit to 4 CPUs to simulate a constrained environment in Colab
ray.init(num_cpus=4, logging_level="WARNING")
print(f"Ray version: {ray.__version__}")
print(f"Available resources: {ray.available_resources()}")

Ray version: 2.54.0
Available resources: {'CPU': 4.0, 'object_store_memory': 3947620761.0, 'node:172.28.0.12': 1.0, 'memory': 9211115111.0, 'node:__internal_head__': 1.0}


/usr/local/lib/python3.12/dist-packages/ray/_private/worker.py:2052: FutureWarning: Tip: In future versions of Ray, Ray will no longer override accelerator visible devices env var if num_gpus=0 or num_gpus=None (default). To enable this behavior and turn off this error message, set RAY_ACCEL_ENV_VAR_OVERRIDE_ON_ZERO=0
  warnings.warn(


In [ ]:
### 1. TASKS: Parallel Execution & Dependencies ###

# The @ray.remote decorator turns a standard function into a distributed task
@ray.remote
def add(a, b):
    time.sleep(0.2)  # Simulate work
    return a + b

@ray.remote
def multiply(a, b):
    time.sleep(0.2)
    return a * b

print("Building task graph...")
# .remote() submits the task asynchronously and returns an ObjectRef immediately
x = add.remote(1, 2)
y = add.remote(3, 4)
z = multiply.remote(x, y)    # Ray automatically handles dependencies (waits for x and y)

print("Executing (tasks run in parallel where possible)...")
start = time.time()
# ray.get() blocks until the result is ready
result = ray.get(z)
elapsed = time.time() - start

print(f"Result: (1+2) * (3+4) = {result}")
print(f"Time: {elapsed:.2f}s (two 0.2s tasks in parallel + one 0.2s task)")

Building task graph...
Executing (tasks run in parallel where possible)...
Result: (1+2) * (3+4) = 21
Time: 0.78s (two 0.2s tasks in parallel + one 0.2s task)


In [ ]:
### 2. ACTORS: Stateful Processing ###

# Actors are perfect for things that need to maintain state, like model serving or tracking metrics.
@ray.remote
class MetricsTracker:
    def __init__(self):
        self.counter = 0

    def update(self):
        self.counter += 1
        return self.counter

# Create an actor instance on a worker
tracker = MetricsTracker.remote()

# Call methods on the actor sequentially
print("Update 1:", ray.get(tracker.update.remote()))
print("Update 2:", ray.get(tracker.update.remote()))

Update 1: 1
Update 2: 2


In [ ]:
### 3. OBJECT STORE: Zero-Copy Memory Management ###

# Passing large arrays directly to tasks causes expensive serialization.
# The Object Store allows workers to share memory efficiently.

@ray.remote
def compute_mean(data):
    return float(np.mean(data))

# Create a large array (~7.6 MB)
large_array = np.random.rand(1_000_000)

# ray.put() stores the data in Ray's shared memory once
array_ref = ray.put(large_array)

# We pass the lightweight reference instead of the heavy data
future = compute_mean.remote(array_ref)
print(f"Mean of large array: {ray.get(future):.4f}")

Mean of large array: 0.4998


---
## Part 2: Ray Data (Distributed Datasets)

Unlike Spark DataFrames which are heavily optimized for SQL and ETL pipelines, Ray Datasets are optimized for **machine learning preprocessing and tensor shuffling**.

The star of the show here is `map_batches()`, which applies transformations to chunks of data simultaneously.

In [ ]:
# 1. Generate synthetic weather data
np.random.seed(42)
n_samples = 20000

weather_df = pd.DataFrame({
    "temperature": np.random.uniform(0, 35, n_samples),
    "humidity": np.random.uniform(20, 100, n_samples),
    "wind_speed": np.random.uniform(0, 30, n_samples),
    "target_precipitation": np.random.exponential(1.5, n_samples)
})

# 2. Convert to a distributed Ray Dataset
ds_weather = ray.data.from_pandas(weather_df)
print(f"Created Ray Dataset with {ds_weather.count()} rows.")

Created Ray Dataset with 20000 rows.


In [ ]:
# 3. Vectorized feature engineering using map_batches()
def engineer_features(batch: pd.DataFrame) -> pd.DataFrame:
    """This function applies transformations to entire chunks of data at once."""
    result = batch.copy()

    # Create a synthetic interaction feature: T_celsius + (Humidity * 0.1)
    result["temp_humidity_idx"] = result["temperature"] + (result["humidity"] * 0.1)
    return result

ds_processed = ds_weather.map_batches(engineer_features, batch_format="pandas")

# 4. Train/Test Split
train_ds, valid_ds = ds_processed.train_test_split(test_size=0.2)

print(f"Training rows: {train_ds.count()} | Validation rows: {valid_ds.count()}")
print("\nSample processed row:")
train_ds.take(1)

2026-03-04 00:40:56,379	INFO logging.py:392 -- Registered dataset logger for dataset dataset_2_0
2026-03-04 00:40:56,400	INFO streaming_executor.py:182 -- Starting execution of Dataset dataset_2_0. Full logs are in /tmp/ray/session_2026-03-04_00-39-55_703859_13480/logs/ray-data
2026-03-04 00:40:56,401	INFO streaming_executor.py:183 -- Execution plan of Dataset dataset_2_0: InputDataBuffer[Input] -> TaskPoolMapOperator[MapBatches(engineer_features)->Project] -> AggregateNumRows[AggregateNumRows]
2026-03-04 00:40:56,405	WARNING resource_manager.py:141 -- ⚠️  Ray's object store is configured to use only 42.9% of available memory (3.7GiB out of 8.6GiB total). For optimal Ray Data performance, we recommend setting the object store to at least 50% of available memory. You can do this by setting the 'object_store_memory' parameter when calling ray.init() or by setting the RAY_DEFAULT_OBJECT_STORE_MEMORY_PROPORTION environment variable.
2026-03-04 00:40:56,409	INFO __init__.py:56 -- Progress w

Training rows: 16000 | Validation rows: 4000

Sample processed row:


[{'temperature': 13.108904159657687,
  'humidity': 78.39986487919762,
  'wind_speed': 8.967361236132495,
  'target_precipitation': 2.029609739326851,
  'temp_humidity_idx': 20.94889064757745}]



---
## Part 3: Ray Train (Distributed Model Training)

Now we pass our distributed Ray Datasets directly into a distributed trainer. Ray handles the resource allocation, data shuffling, and worker coordination without us having to write complex boilerplate code.

In [ ]:
import xgboost as xgb
import ray.train
from ray.train.xgboost import XGBoostTrainer, RayTrainReportCallback
from ray.train import ScalingConfig

def train_loop_per_worker(config: dict):
    # 1. Fetch the data shard for this specific worker
    train_ds_iter = ray.train.get_dataset_shard("train")
    valid_ds_iter = ray.train.get_dataset_shard("valid")

    # 2. Materialize to Pandas DataFrames
    train_df = train_ds_iter.materialize().to_pandas()
    valid_df = valid_ds_iter.materialize().to_pandas()

    # 3. Split features and labels
    label_col = "target_precipitation"
    train_X, train_y = train_df.drop(columns=[label_col]), train_df[label_col]
    valid_X, valid_y = valid_df.drop(columns=[label_col]), valid_df[label_col]

    # 4. Convert to XGBoost DMatrix
    dtrain = xgb.DMatrix(train_X, label=train_y)
    dvalid = xgb.DMatrix(valid_X, label=valid_y)

    # 5. Run standard XGBoost training with Ray's callback
    xgb.train(
        params=config,
        dtrain=dtrain,
        evals=[(dvalid, "valid")],
        num_boost_round=50,
        callbacks=[RayTrainReportCallback()]
    )

# Define the distributed trainer using the new API
trainer = XGBoostTrainer(
    train_loop_per_worker=train_loop_per_worker,
    train_loop_config={
        "objective": "reg:squarederror",
        "eval_metric": "rmse",
        "max_depth": 6,
        "eta": 0.1,
    },
    datasets={"train": train_ds, "valid": valid_ds},
    scaling_config=ScalingConfig(
        num_workers=2,
        use_gpu=False,
    ),
)

print("Starting distributed training...")
result = trainer.fit()

print("\nTraining Complete!")
print(f"Final Validation RMSE: {result.metrics.get('valid-rmse', 'N/A'):.4f}")
print(f"Checkpoint saved at: {result.checkpoint}")

Starting distributed training...


(TrainController pid=14578) Attempting to start training worker group of size 2 with the following resources: [{'CPU': 1}] * 2


(raylet) WARNING: 8 PYTHON worker processes have been started on node: d60d97da09ee221b24085bdd8b965a971909e2c1c589f1663837ea44 with address: 172.28.0.12. This could be a result of using a large number of actors, or due to tasks blocked in ray.get() calls (see https://github.com/ray-project/ray/issues/3644 for some discussion of workarounds).
(raylet) WARNING: 12 PYTHON worker processes have been started on node: d60d97da09ee221b24085bdd8b965a971909e2c1c589f1663837ea44 with address: 172.28.0.12. This could be a result of using a large number of actors, or due to tasks blocked in ray.get() calls (see https://github.com/ray-project/ray/issues/3644 for some discussion of workarounds).


(TrainController pid=14578) Started training worker group of size 2: 
(TrainController pid=14578) - (ip=172.28.0.12, pid=14742) world_rank=0, local_rank=0, node_rank=0
(TrainController pid=14578) - (ip=172.28.0.12, pid=14769) world_rank=1, local_rank=1, node_rank=0
(RayTrainWorker pid=14742) [00:41:48] Task [xgboost.ray-rank=00000000]:9497c021bbe412360159263b01000000 got rank 0
(SplitCoordinator pid=14916) Registered dataset logger for dataset train_6_0
(SplitCoordinator pid=14916) Starting execution of Dataset train_6_0. Full logs are in /tmp/ray/session_2026-03-04_00-39-55_703859_13480/logs/ray-data
(SplitCoordinator pid=14916) Execution plan of Dataset train_6_0: InputDataBuffer[Input] -> OutputSplitter[split(2, equal=True)]
(SplitCoordinator pid=14916) ⚠️  Ray's object store is configured to use only 42.9% of available memory (3.7GiB out of 8.6GiB total). For optimal Ray Data performance, we recommend setting the object store to at least 50% of available memory. You can do this by 

(pid=14916) Running Dataset train_6_0.: 0.00 row [00:00, ? row/s]

...

(pid=14742) Running Dataset dataset_13_0.: 0.00 row [00:00, ? row/s]

(RayTrainWorker pid=14742) 


(pid=14769) Running Dataset dataset_12_0.: 0.00 row [00:00, ? row/s]

(RayTrainWorker pid=14769) 
(RayTrainWorker pid=14742) 
(RayTrainWorker pid=14769) 
(TrainController pid=14578) [00:41:57] [0]	valid-rmse:1.46453
(RayTrainWorker pid=14742) Reporting training result 1: TrainingReport(checkpoint=None, metrics=OrderedDict({'valid-rmse': np.float64(1.4645270983382617)}), validation=False)
(RayTrainWorker pid=14742) Reporting training result 2: TrainingReport(checkpoint=None, metrics=OrderedDict({'valid-rmse': np.float64(1.464516309997357)}), validation=False)
(TrainController pid=14578) [00:41:58] [1]	valid-rmse:1.46452
(TrainController pid=14578) [00:42:00] [2]	valid-rmse:1.46502
(TrainController pid=14578) [00:42:02] [3]	valid-rmse:1.46503
(RayTrainWorker pid=14769) Registered dataset logger for dataset dataset_12_0 [repeated 2x across cluster]
(SplitCoordinator pid=14917) ⚠️  Ray's object store is configured to use only 42.9% of available memory (3.7GiB out of 8.6GiB total). For optimal Ray Data performance, we recommend setting the object store to at 


Training Complete!
Final Validation RMSE: 1.4726
Checkpoint saved at: Checkpoint(filesystem=local, path=/root/ray_results/ray_train_run-2026-03-04_00-41-01/checkpoint_2026-03-04_00-43-35.006012)


---
## Part 4: Interactive Exercise - Your Turn!

**Task:** Build a quick pipeline to predict `wind_speed` instead of `precipitation`.
1. Use the original `ds_weather` dataset.
2. Drop the `target_precipitation` column.
3. Split the data.
4. Train an `XGBoostTrainer` using `wind_speed` as the label.

*(Hint: `ds_weather.drop_columns(["target_precipitation"])` is your friend here!)*

In [ ]:
# Students: Write your pipeline here!

# -----------------
# SOLUTION BELOW
# -----------------
# """
import xgboost as xgb
import ray.train
from ray.train.xgboost import XGBoostTrainer, RayTrainReportCallback
from ray.train import ScalingConfig

# 1. Prepare data (Drop the original target)
wind_ds = ds_weather.drop_columns(["target_precipitation"])

# 2. Split
train_wind, valid_wind = wind_ds.train_test_split(test_size=0.2)

# 3. Define the custom training loop for the workers
def wind_train_loop_per_worker(config: dict):
    train_ds_iter = ray.train.get_dataset_shard("train")
    valid_ds_iter = ray.train.get_dataset_shard("valid")

    train_df = train_ds_iter.materialize().to_pandas()
    valid_df = valid_ds_iter.materialize().to_pandas()

    label_col = "wind_speed"
    train_X, train_y = train_df.drop(columns=[label_col]), train_df[label_col]
    valid_X, valid_y = valid_df.drop(columns=[label_col]), valid_df[label_col]

    dtrain = xgb.DMatrix(train_X, label=train_y)
    dvalid = xgb.DMatrix(valid_X, label=valid_y)

    xgb.train(
        params=config,
        dtrain=dtrain,
        evals=[(dvalid, "valid")],
        num_boost_round=30,
        callbacks=[RayTrainReportCallback()]
    )

# 4. Train using the new API
wind_trainer = XGBoostTrainer(
    train_loop_per_worker=wind_train_loop_per_worker,
    train_loop_config={
        "objective": "reg:squarederror",
        "eval_metric": "rmse",
        "max_depth": 4,
    },
    datasets={"train": train_wind, "valid": valid_wind},
    scaling_config=ScalingConfig(num_workers=2, use_gpu=False),
)

wind_result = wind_trainer.fit()
print(f"Wind Prediction RMSE: {wind_result.metrics.get('valid-rmse'):.4f}")
# """

2026-03-04 00:43:41,755	INFO logging.py:392 -- Registered dataset logger for dataset dataset_15_0
2026-03-04 00:43:41,764	INFO streaming_executor.py:182 -- Starting execution of Dataset dataset_15_0. Full logs are in /tmp/ray/session_2026-03-04_00-39-55_703859_13480/logs/ray-data
2026-03-04 00:43:41,766	INFO streaming_executor.py:183 -- Execution plan of Dataset dataset_15_0: InputDataBuffer[Input] -> TaskPoolMapOperator[MapBatches(drop_columns)->Project] -> AggregateNumRows[AggregateNumRows]
2026-03-04 00:43:41,910	INFO logging_progress.py:174 -- ======= Running Dataset: dataset_15_0 =======
2026-03-04 00:43:41,912	INFO logging_progress.py:225 -- Total Progress: 0/?
2026-03-04 00:43:41,916	INFO logging_progress.py:227 -- Active & requested resources: 0/4 CPU, 0.0B/1.8GiB object store
2026-03-04 00:43:41,919	INFO logging_progress.py:181 -- 
2026-03-04 00:43:41,922	INFO logging_progress.py:231 -- MapBatches(drop_columns)->Project: 0/1
2026-03-04 00:43:41,924	INFO logging_progress.py:233

(pid=16287) Running Dataset train_18_0.: 0.00 row [00:00, ? row/s]

(pid=16287) - split(2, equal=True):   0%|          | 0.00/1.00 [00:00<?, ? row/s]

(SplitCoordinator pid=16287) Registered dataset logger for dataset train_18_0
(SplitCoordinator pid=16287) Starting execution of Dataset train_18_0. Full logs are in /tmp/ray/session_2026-03-04_00-39-55_703859_13480/logs/ray-data
(SplitCoordinator pid=16287) Execution plan of Dataset train_18_0: InputDataBuffer[Input] -> OutputSplitter[split(2, equal=True)]
(SplitCoordinator pid=16287) ⚠️  Ray's object store is configured to use only 42.9% of available memory (3.7GiB out of 8.6GiB total). For optimal Ray Data performance, we recommend setting the object store to at least 50% of available memory. You can do this by setting the 'object_store_memory' parameter when calling ray.init() or by setting the RAY_DEFAULT_OBJECT_STORE_MEMORY_PROPORTION environment variable.
(SplitCoordinator pid=16287) [dataset]: A new progress UI is available. To enable, set `ray.data.DataContext.get_current().enable_rich_progress_bars = True` and `ray.data.DataContext.get_current().use_ray_tqdm = False`.
(SplitC

(pid=16115) Running Dataset dataset_23_0.: 0.00 row [00:00, ? row/s]

...

(pid=16115) Running Dataset dataset_24_0.: 0.00 row [00:00, ? row/s]

(RayTrainWorker pid=16115) 
(RayTrainWorker pid=16115) 


(pid=16144) Running Dataset dataset_25_0.: 0.00 row [00:00, ? row/s]

(RayTrainWorker pid=16144) 
(RayTrainWorker pid=16144) 
(TrainController pid=15948) [00:44:28] [0]	valid-rmse:8.67445
(RayTrainWorker pid=16115) Reporting training result 1: TrainingReport(checkpoint=None, metrics=OrderedDict({'valid-rmse': np.float64(8.674450316530901)}), validation=False)
(RayTrainWorker pid=16115) Reporting training result 2: TrainingReport(checkpoint=None, metrics=OrderedDict({'valid-rmse': np.float64(8.673004587833603)}), validation=False)
(TrainController pid=15948) [00:44:29] [1]	valid-rmse:8.67300
(TrainController pid=15948) [00:44:31] [2]	valid-rmse:8.67395
(TrainController pid=15948) [00:44:33] [3]	valid-rmse:8.67565
(RayTrainWorker pid=16144) [00:44:27] Task [xgboost.ray-rank=00000001]:9d6b34b62a5fdf3ddec15d3701000000 got rank 1
(RayTrainWorker pid=16144) Registered dataset logger for dataset dataset_25_0 [repeated 5x across cluster]
(SplitCoordinator pid=16288) Starting execution of Dataset valid_20_0. Full logs are in /tmp/ray/session_2026-03-04_00-39-55_7

Wind Prediction RMSE: 8.7106


---
## Summary
1. Use **Ray Tasks (`@ray.remote`)** for stateless parallel computing.
2. Use **Ray Actors** when you need to maintain state (like serving a model).
3. Use **Ray Data (`map_batches`)** instead of Spark when prepping data for ML models—it's optimized for tensor formats.
4. Use **Ray Train (`ScalingConfig`)** to easily scale models like XGBoost across cluster nodes.

In [ ]:
# Always cleanly shut down your Ray cluster when finished!
ray.shutdown()
print("Cluster shut down successfully.")

Cluster shut down successfully.
